# MAXIM S-2 baseline (No MoE)

Baseline multitarea comparable con MAXIM+MoE: mismo dataset ampliado, inicialización aleatoria, presupuesto, augmentations, muestreo uniforme, supervisión multietapa normalizada y validación exhaustiva determinista. El sanity check está habilitado; el entrenamiento largo requiere `START_TRAINING=True`.


In [ ]:
%cd /content
!git clone https://github.com/Matiata/maxim.git
%cd /content/maxim
!pip install -r requirements.txt
!pip install -e .
%pip install -q --upgrade "jax[cuda12]==0.10.2" "flax==0.11.2"

/content
Cloning into 'maxim'...
remote: Enumerating objects: 524, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 524 (delta 74), reused 82 (delta 52), pack-reused 408 (from 1)
Receiving objects: 100% (524/524), 39.23 MiB | 20.28 MiB/s, done.
Resolving deltas: 100% (297/297), done.
/content/maxim
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: toolz
    Found existing installation: toolz 0.12.1
    Uninstalling toolz-0.12.1:
      Successfully uninstalled toolz-0.12.1
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
   

In [ ]:
from google.colab import drive # works only for colab
drive.mount('/content/gdrive/',)

Mounted at /content/gdrive/


In [ ]:
import collections
import functools
import importlib
import io
import json
import sys
import os
import re
import time
from typing import Any

import jax
import jax.numpy as jnp
from jax import random
import ml_collections
import numpy as np
import optax
from PIL import Image
import tensorflow as tf
from flax import traverse_util
from flax.core import freeze, unfreeze
from flax.training import train_state, checkpoints

# Runtime configuration

In [ ]:
TRAIN_MODE = "multi_task"  # comparación principal
TASK = "enhance"            # Used only when TRAIN_MODE == "single_task"

TASKS = ["deblur", "dehaze", "denoise", "derain", "enhance"]

DATA_ROOT_DIR = "/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier"
OUTPUT_ROOT_DIR = "/content/gdrive/MyDrive/Facultad/tesis/ckpts/maxim_no_moe"

BATCH_SIZE = 2
EVAL_BATCH_SIZE = 1
NUM_EPOCHS = 30
LEARNING_RATE = 2e-4
WARMUP_EPOCHS = 3
PATCH_SIZE = 256
# Eval en resolución completa puede provocar OOM con MAXIM.
# Usar PATCH_SIZE para validar por parches (recomendado) o None para full-res.
EVAL_PATCH_SIZE = PATCH_SIZE
LOG_EVERY = 10
SAVE_EVERY = 2
WEIGHT_DECAY = 1e-4
SEED = 42

SAMPLING_MODE_TRAIN = "uniform"
STEPS_PER_EPOCH_OVERRIDE = 2000  # None usa el total proporcional.
AUXILIARY_LOSS_WEIGHT = 1.0

# MULTITASK_INIT_CKPT = "/content/gdrive/MyDrive/Facultad/tesis/ckpts/ckpt_Enhancement_LOL.npz"  # warm-start (corrida original)
MULTITASK_INIT_CKPT = ""  # from-scratch multi_task (random init) -- comparacion justa vs MoE
MULTITASK_VARIANT = "S-2"  # escala del backbone multi-task: S-2 = misma que MoE (comparacion justa)

ALLOW_OVERWRITE_EXISTING_RUN = False

SANITY_STEPS = 2
START_TRAINING = True

_MODEL_CONFIGS = {
    "variant": "",
    "dropout_rate": 0.1,
    "num_outputs": 3,
    "use_bias": True,
    "num_supervision_scales": 3,
}
_MODEL_VARIANT_DICT = {
    "denoise": "S-3",
    "deblur": "S-3",
    "derain": "S-2",
    "dehaze": "S-2",
    "enhance": "S-2",
}

TASK_DIR_MAP = {task: os.path.join(DATA_ROOT_DIR, task) for task in TASKS}

if TRAIN_MODE not in {"single_task", "multi_task"}:
    raise ValueError(f"Unknown TRAIN_MODE: {TRAIN_MODE}")
if TASK not in TASK_DIR_MAP:
    raise ValueError(f"Unknown TASK: {TASK}")

if TRAIN_MODE == "single_task":
    ACTIVE_VARIANT = _MODEL_VARIANT_DICT[TASK]
    ACTIVE_TASK_DIRS = [TASK_DIR_MAP[TASK]]
    RUN_NAME = f"single_task_{TASK}"
else:
    ACTIVE_VARIANT = MULTITASK_VARIANT
    ACTIVE_TASK_DIRS = [TASK_DIR_MAP[t] for t in TASKS]
    init_tag = "scratch" if not MULTITASK_INIT_CKPT else "warm"
    RUN_NAME = f"multi_task_all_{ACTIVE_VARIANT}_{init_tag}"

OUTPUT_DIR = os.path.join(OUTPUT_ROOT_DIR, RUN_NAME)
BEST_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "best_checkpoint")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(BEST_OUTPUT_DIR, exist_ok=True)

print("=== Configuration ===")
print(f"TRAIN_MODE: {TRAIN_MODE}")
print(f"TASK: {TASK}")
print(f"ACTIVE_VARIANT: {ACTIVE_VARIANT}")
print(f"TASK_DIRS: {ACTIVE_TASK_DIRS}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"MULTITASK_INIT_CKPT: {MULTITASK_INIT_CKPT}")
TRAIN_LOG_PATH = os.path.join(OUTPUT_DIR, "training.log")
np.random.seed(SEED)
tf.random.set_seed(SEED)


=== Configuration ===
TRAIN_MODE: multi_task
TASK: enhance
ACTIVE_VARIANT: S-2
TASK_DIRS: ['/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur', '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze', '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise', '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/derain', '/content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/enhance']
OUTPUT_DIR: /content/gdrive/MyDrive/Facultad/tesis/ckpts/maxim_no_moe/multi_task_all_S-2_scratch
MULTITASK_INIT_CKPT: 


# Auxiliar Functions


In [ ]:
def recover_tree(keys, values):
    """Recover nested dict from flat checkpoint keys."""
    tree = {}
    sub_trees = collections.defaultdict(list)
    for k, v in zip(keys, values):
        if "/" not in k:
            tree[k] = v
        else:
            k_left, k_right = k.split("/", 1)
            sub_trees[k_left].append((k_right, v))
    for k, kv_pairs in sub_trees.items():
        k_subtree, v_subtree = zip(*kv_pairs)
        tree[k] = recover_tree(k_subtree, v_subtree)
    return tree


def to_frozen_params(params):
    """Normalize params to FrozenDict to keep JAX treedef stable."""
    return freeze(unfreeze(params))


def reset_optimizer_with_params(state, params):
    """Rebuild optimizer state so it matches loaded param structure."""
    params = to_frozen_params(params)
    step_dtype = jnp.asarray(state.step).dtype
    step0 = jnp.asarray(0, dtype=step_dtype)
    return state.replace(params=params, opt_state=state.tx.init(params), step=step0)


def get_npz_params(ckpt_path):
    """Load MAXIM params from original .npz checkpoint format."""
    with tf.io.gfile.GFile(ckpt_path, "rb") as f:
        data = f.read()
    values = np.load(io.BytesIO(data), allow_pickle=False)
    params = recover_tree(*zip(*values.items()))
    if "opt" in params and "target" in params["opt"]:
        return params["opt"]["target"]
    return params


def load_matching_params(target_params, source_params):
    """Load only matching params and report coverage."""
    flat_target = traverse_util.flatten_dict(unfreeze(target_params))
    flat_source = traverse_util.flatten_dict(source_params)

    updated = dict(flat_target)
    loaded = 0
    skipped_shape = 0

    for key, value in flat_source.items():
        if key in flat_target:
            if flat_target[key].shape == value.shape:
                updated[key] = value
                loaded += 1
            else:
                skipped_shape += 1

    merged = freeze(traverse_util.unflatten_dict(updated))
    return merged, {"loaded": loaded, "skipped_shape": skipped_shape, "total_target": len(flat_target)}


def initialize_state_by_mode(state):
    """Apply requested init policy:
    - single_task: keep random init
    - multi_task: load manual checkpoint path, or random init if MULTITASK_INIT_CKPT is empty
    """
    if TRAIN_MODE == "single_task":
        print("single_task mode: using random initialization.")
        return reset_optimizer_with_params(state, state.params)

    if not MULTITASK_INIT_CKPT:
        print("multi_task mode: MULTITASK_INIT_CKPT empty -> using random initialization.")
        return reset_optimizer_with_params(state, state.params)

    ckpt_path = MULTITASK_INIT_CKPT
    if ckpt_path.endswith(".npz"):
        print(f"Loading multi-task init from NPZ: {ckpt_path}")
        source_params = get_npz_params(ckpt_path)
        new_params, report = load_matching_params(state.params, source_params)
        if report["loaded"] == 0:
            raise ValueError(
                "No compatible parameters were loaded from NPZ checkpoint. "
                "Check variant and checkpoint path."
            )
        print(
            f"Loaded {report['loaded']} / {report['total_target']} params "
            f"(shape-skipped: {report['skipped_shape']})."
        )
        return reset_optimizer_with_params(state, new_params)

    print(f"Loading multi-task init from Flax checkpoint dir/file: {ckpt_path}")
    try:
        restored = checkpoints.restore_checkpoint(ckpt_dir=ckpt_path, target=state)
    except Exception as err:
        raise ValueError(
            f"Failed to restore multi-task checkpoint from '{ckpt_path}': {err}"
        ) from err

    if restored is None:
        raise ValueError(f"Checkpoint path '{ckpt_path}' did not return a valid state.")
    if not hasattr(restored, "params"):
        raise ValueError(
            "Restored object does not expose 'params'. "
            f"Got type: {type(restored)}"
        )

    print("Multi-task checkpoint restored successfully.")
    return reset_optimizer_with_params(restored, restored.params)

def _checkpoint_epoch(checkpoint_path):
    match = re.search(r"checkpoint_(\d+)$", checkpoint_path.rstrip("/"))
    return int(match.group(1)) if match else None


def resolve_resume_checkpoint(output_dir, resume_mode, resume_epoch):
    """Resolve an exact checkpoint file and fail instead of silently starting over."""
    if resume_mode == "latest":
        checkpoint_path = checkpoints.latest_checkpoint(output_dir)
    elif resume_mode == "best":
        checkpoint_path = checkpoints.latest_checkpoint(
            os.path.join(output_dir, "best_checkpoint")
        )
    elif resume_mode == "specific":
        checkpoint_path = os.path.join(output_dir, f"checkpoint_{resume_epoch}")
    else:
        raise ValueError(f"Invalid RESUME_MODE: {resume_mode}")

    if not checkpoint_path or not tf.io.gfile.exists(checkpoint_path):
        raise FileNotFoundError(
            f"No checkpoint found for RESUME_MODE={resume_mode!r} in {output_dir}."
        )
    return checkpoint_path


def restore_training_state_or_fail(state, checkpoint_path, steps_per_epoch):
    """Restore params, batch_stats, optimizer moments and the real step counter."""
    step_before = int(jax.device_get(state.step))
    restored = checkpoints.restore_checkpoint(
        ckpt_dir=checkpoint_path, target=state
    )
    restored_step = int(jax.device_get(restored.step))
    if restored_step == step_before:
        raise FileNotFoundError(
            f"restore_checkpoint loaded nothing from {checkpoint_path}."
        )

    completed_epochs, partial_steps = divmod(restored_step, steps_per_epoch)
    if partial_steps:
        raise ValueError(
            f"Checkpoint step {restored_step} is not an epoch boundary for "
            f"{steps_per_epoch} steps/epoch."
        )
    filename_epoch = _checkpoint_epoch(checkpoint_path)
    if filename_epoch is not None and filename_epoch != completed_epochs:
        raise ValueError(
            f"Checkpoint filename says epoch {filename_epoch}, but state.step "
            f"corresponds to epoch {completed_epochs}."
        )
    return restored, completed_epochs


def load_historical_best_psnr(output_dir):
    """Load the validated best score so resume cannot overwrite it with a worse model."""
    metric_path = os.path.join(output_dir, "best_checkpoint", "best_metric.json")
    if not tf.io.gfile.exists(metric_path):
        raise FileNotFoundError(
            f"Missing {metric_path}; refusing to resume with an unknown best PSNR."
        )
    with tf.io.gfile.GFile(metric_path, "r") as handle:
        metric = json.load(handle)
    best_psnr = metric.get("weighted_psnr", metric.get("psnr"))
    if best_psnr is None or not np.isfinite(float(best_psnr)):
        raise ValueError(f"Invalid best PSNR metadata in {metric_path}: {metric}")
    return float(best_psnr), int(metric.get("epoch", -1))


In [ ]:
def resize_to_match(image, target):
    """Pad/crop image to match target dimensions."""
    h, w = image.shape[0], image.shape[1]
    th, tw = target.shape[0], target.shape[1]
    if h == tw and w == th:
        image = np.rot90(image)
        h, w = image.shape[:2]
    pad_h = max(0, th - h)
    pad_w = max(0, tw - w)
    if pad_h > 0 or pad_w > 0:
        image = np.pad(image, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
        h, w = image.shape[:2]
    if h > th or w > tw:
        top = max(0, (h - th) // 2)
        left = max(0, (w - tw) // 2)
        image = image[top:top + th, left:left + tw]
    return image


def random_crop(image, target, crop_size):
    h_i, w_i = image.shape[:2]
    h_t, w_t = target.shape[:2]
    if h_i < crop_size or w_i < crop_size:
        image = np.pad(
            image,
            ((0, max(0, crop_size - h_i)), (0, max(0, crop_size - w_i)), (0, 0)),
            mode="reflect",
        )
    if h_t < crop_size or w_t < crop_size:
        target = np.pad(
            target,
            ((0, max(0, crop_size - h_t)), (0, max(0, crop_size - w_t)), (0, 0)),
            mode="reflect",
        )
    h = min(image.shape[0], target.shape[0])
    w = min(image.shape[1], target.shape[1])
    top = np.random.randint(0, h - crop_size + 1)
    left = np.random.randint(0, w - crop_size + 1)
    return (
        image[top:top + crop_size, left:left + crop_size],
        target[top:top + crop_size, left:left + crop_size],
    )


def center_crop(image, target, crop_size):
    h_i, w_i = image.shape[:2]
    h_t, w_t = target.shape[:2]
    if h_i < crop_size or w_i < crop_size:
        image = np.pad(
            image,
            ((0, max(0, crop_size - h_i)), (0, max(0, crop_size - w_i)), (0, 0)),
            mode="reflect",
        )
    if h_t < crop_size or w_t < crop_size:
        target = np.pad(
            target,
            ((0, max(0, crop_size - h_t)), (0, max(0, crop_size - w_t)), (0, 0)),
            mode="reflect",
        )
    h = min(image.shape[0], target.shape[0])
    w = min(image.shape[1], target.shape[1])
    top = max(0, (h - crop_size) // 2)
    left = max(0, (w - crop_size) // 2)
    return (
        image[top:top + crop_size, left:left + crop_size],
        target[top:top + crop_size, left:left + crop_size],
    )


def random_flip(image, target):
    if np.random.rand() > 0.5:
        image, target = np.fliplr(image), np.fliplr(target)
    if np.random.rand() > 0.5:
        image, target = np.flipud(image), np.flipud(target)
    return image, target


def random_rotation(image, target):
    k = np.random.randint(0, 4)
    return np.rot90(image, k=k), np.rot90(target, k=k)


def set_shapes(inp, tgt, sizes):
    inp.set_shape([PATCH_SIZE, PATCH_SIZE, 3])
    tgt.set_shape([PATCH_SIZE, PATCH_SIZE, 3])
    sizes.set_shape([6])
    return inp, tgt, sizes


def read_lines_from_file(basepath, filepath):
    with open(filepath, "r", encoding="utf-8") as handle:
        lines = [line.strip() for line in handle if line.strip()]
    existing = []
    missing = []
    for name in lines:
        path = os.path.join(basepath, name)
        (existing if os.path.exists(path) else missing).append(path)
    print(
        f"Checked {len(lines)} files in {basepath}: "
        f"{len(existing)} existing, {len(missing)} missing."
    )
    if missing:
        print(f"Missing sample: {missing[:5]}")
    return existing, missing


def load_image(filepath, max_retries=5, retry_delay=0.2):
    """Load an RGB image, retrying transient Google Drive I/O errors."""
    last_error = None
    for attempt in range(max_retries):
        try:
            with Image.open(filepath) as image:
                return np.asarray(image.convert("RGB"), np.float32) / 255.0
        except OSError as error:
            last_error = error
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
    raise OSError(f"Failed to read image after {max_retries} attempts: {filepath}") from last_error


def create_dataset(data_dir, batch_size, patch_size, is_training=True):
    print(f"Creating {'training' if is_training else 'validation'} dataset from {data_dir}")
    input_dir = os.path.join(data_dir, "imgs")
    target_dir = os.path.join(data_dir, "GT")
    files_list = os.path.join(data_dir, "train.txt" if is_training else "test.txt")
    input_files, missing_inputs = read_lines_from_file(input_dir, files_list)
    target_files, missing_targets = read_lines_from_file(target_dir, files_list)
    if missing_inputs or missing_targets:
        raise FileNotFoundError(f"Dataset lists contain missing files in {data_dir}.")
    if len(input_files) != len(target_files):
        raise ValueError(f"Input/GT count mismatch in {data_dir}.")

    def load_and_preprocess(input_path, target_path):
        inp = load_image(input_path.numpy().decode())
        tgt = load_image(target_path.numpy().decode())
        orig_h, orig_w = inp.shape[:2]
        if is_training:
            inp, tgt = random_crop(inp, tgt, patch_size)
            inp, tgt = random_flip(inp, tgt)
            inp, tgt = random_rotation(inp, tgt)
        else:
            inp, tgt = center_crop(inp, tgt, patch_size)
        if inp.shape != (patch_size, patch_size, 3) or tgt.shape != inp.shape:
            raise ValueError(
                f"Invalid processed shapes for {input_path.numpy().decode()}: "
                f"input={inp.shape}, GT={tgt.shape}."
            )
        sizes = np.array(
            [orig_h, orig_w, patch_size, patch_size, patch_size, patch_size],
            dtype=np.int32,
        )
        return inp.astype(np.float32), tgt.astype(np.float32), sizes

    dataset = tf.data.Dataset.from_tensor_slices((input_files, target_files))
    if is_training:
        dataset = dataset.shuffle(
            buffer_size=1000, seed=SEED, reshuffle_each_iteration=True
        )
    parallel_reads = min(4, os.cpu_count() or 1)
    dataset = dataset.map(
        lambda x, y: tf.py_function(
            func=load_and_preprocess,
            inp=[x, y],
            Tout=[tf.float32, tf.float32, tf.int32],
        ),
        num_parallel_calls=parallel_reads,
        deterministic=True,
    )
    dataset = dataset.map(
        set_shapes, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True
    )
    dataset = dataset.batch(batch_size, drop_remainder=is_training)
    return dataset.prefetch(tf.data.AUTOTUNE), len(input_files)


def create_dataset_unbatched(data_dir, patch_size, is_training=True):
    dataset, length = create_dataset(
        data_dir=data_dir,
        batch_size=1,
        patch_size=patch_size,
        is_training=is_training,
    )
    return dataset.unbatch(), length


def add_task_id(dataset, task_id):
    task_id = tf.constant(task_id, dtype=tf.int32)
    return dataset.map(
        lambda x, y, sizes: (x, y, sizes, task_id),
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=True,
    )


def create_unified_dataset(
    task_dirs,
    batch_size,
    patch_size,
    is_training=True,
    sampling_mode="uniform",
    shuffle_buffer=1000,
):
    datasets = []
    lengths = []
    for task_id, data_dir in enumerate(task_dirs):
        dataset, length = create_dataset_unbatched(
            data_dir=data_dir,
            patch_size=patch_size,
            is_training=is_training,
        )
        datasets.append(add_task_id(dataset, task_id))
        lengths.append(length)

    total_samples = sum(lengths)
    if is_training:
        lengths_tensor = tf.constant(lengths, dtype=tf.float32)
        if sampling_mode == "uniform":
            weights = tf.ones_like(lengths_tensor) / tf.cast(
                tf.size(lengths_tensor), tf.float32
            )
        elif sampling_mode == "proportional":
            weights = lengths_tensor / tf.reduce_sum(lengths_tensor)
        else:
            raise ValueError(f"Unknown sampling_mode: {sampling_mode}")
        # Repeat each source before mixing so small tasks never disappear.
        repeated = [dataset.repeat() for dataset in datasets]
        unified = tf.data.Dataset.sample_from_datasets(
            repeated,
            weights=weights,
            seed=SEED,
            stop_on_empty_dataset=False,
        )
        unified = unified.shuffle(
            shuffle_buffer, seed=SEED, reshuffle_each_iteration=True
        )
    else:
        # Deterministic concatenation guarantees one exhaustive validation pass.
        unified = datasets[0]
        for dataset in datasets[1:]:
            unified = unified.concatenate(dataset)

    unified = unified.batch(batch_size, drop_remainder=is_training)
    unified = unified.prefetch(tf.data.AUTOTUNE)
    if not is_training:
        steps_per_epoch = (total_samples + batch_size - 1) // batch_size
    elif STEPS_PER_EPOCH_OVERRIDE is not None:
        steps_per_epoch = int(STEPS_PER_EPOCH_OVERRIDE)
    else:
        steps_per_epoch = max(1, total_samples // batch_size)
    return unified, steps_per_epoch, tuple(lengths)


In [ ]:
class TrainState(train_state.TrainState):
    """Extended train state with optional batch statistics."""
    batch_stats: Any = None


def resize_target_to(pred, target):
    if pred.shape == target.shape:
        return target
    scale_h = target.shape[1] // pred.shape[1]
    scale_w = target.shape[2] // pred.shape[2]
    return target[:, ::scale_h, ::scale_w, :]


def compute_supervision_losses(predictions, targets, num_scales=3, auxiliary_weight=1.0):
    """Match the normalized deep-supervision objective used by MAXIM+MoE."""
    if not isinstance(predictions, (list, tuple)) or not predictions:
        raise ValueError("Expected non-empty multi-stage predictions.")
    final_pred = predictions[-1][-1]
    final_target = resize_target_to(final_pred, targets)
    final_loss = jnp.mean(jnp.abs(final_pred - final_target))
    auxiliary_sum = jnp.zeros_like(final_loss)
    auxiliary_weight_sum = 0.0
    last_stage = len(predictions) - 1
    for stage_index, stage_predictions in enumerate(predictions):
        for scale_index, prediction in enumerate(stage_predictions):
            if stage_index == last_stage and scale_index == len(stage_predictions) - 1:
                continue
            scale_weight = 0.5 ** (num_scales - scale_index - 1)
            target_at_scale = resize_target_to(prediction, targets)
            auxiliary_sum += scale_weight * jnp.mean(jnp.abs(prediction - target_at_scale))
            auxiliary_weight_sum += scale_weight
    denominator = jnp.maximum(
        jnp.asarray(auxiliary_weight_sum, dtype=final_loss.dtype),
        jnp.asarray(1e-8, dtype=final_loss.dtype),
    )
    auxiliary_loss = auxiliary_sum / denominator
    reconstruction_loss = final_loss + auxiliary_weight * auxiliary_loss
    return reconstruction_loss, final_loss, auxiliary_loss, final_pred


def compute_psnr_per_example(pred, target):
    diff = pred * 255.0 - target * 255.0
    mse = jnp.mean(diff ** 2, axis=(1, 2, 3))
    mse = jnp.maximum(mse, 1e-6)
    return 20.0 * jnp.log10(255.0 / jnp.sqrt(mse))


def create_learning_rate_schedule(base_lr, warmup_epochs, total_steps, steps_per_epoch):
    warmup_steps = int(warmup_epochs * steps_per_epoch)
    warmup_fn = optax.linear_schedule(
        init_value=0.0,
        end_value=base_lr,
        transition_steps=max(1, warmup_steps),
    )
    cosine_fn = optax.cosine_decay_schedule(
        init_value=base_lr,
        decay_steps=max(1, total_steps - warmup_steps),
        alpha=1e-6,
    )
    return optax.join_schedules([warmup_fn, cosine_fn], boundaries=[warmup_steps])


def build_model(variant):
    maxim_mod = importlib.import_module("maxim.models.maxim")
    maxim_configs = ml_collections.ConfigDict(_MODEL_CONFIGS)
    maxim_configs.variant = variant
    return maxim_mod.Model(**maxim_configs)


def create_train_state(rng, model, learning_rate_fn, weight_decay):
    dummy_input = jnp.ones([1, PATCH_SIZE, PATCH_SIZE, 3])
    variables = model.init(rng, dummy_input, train=True)
    tx = optax.chain(
        optax.clip_by_global_norm(1.0),
        optax.adamw(learning_rate=learning_rate_fn, weight_decay=weight_decay),
    )
    return TrainState.create(
        apply_fn=model.apply,
        params=variables["params"],
        tx=tx,
        batch_stats=variables.get("batch_stats", None),
    )


@functools.partial(jax.jit, static_argnums=(3,))
def train_step(state, batch_input, batch_target, num_scales, rng):
    def loss_fn(params):
        variables = {"params": params}
        if state.batch_stats is not None:
            variables["batch_stats"] = state.batch_stats
            predictions, updates = state.apply_fn(
                variables,
                batch_input,
                train=True,
                rngs={"dropout": rng},
                mutable=["batch_stats"],
            )
            new_batch_stats = updates["batch_stats"]
        else:
            predictions = state.apply_fn(
                variables, batch_input, train=True, rngs={"dropout": rng}
            )
            new_batch_stats = None
        reconstruction, final_loss, auxiliary_loss, final_pred = compute_supervision_losses(
            predictions, batch_target, num_scales, AUXILIARY_LOSS_WEIGHT
        )
        final_target = resize_target_to(final_pred, batch_target)
        psnr = jnp.mean(compute_psnr_per_example(final_pred, final_target))
        return reconstruction, (final_loss, auxiliary_loss, psnr, new_batch_stats)

    (loss, (final_loss, auxiliary_loss, psnr, new_batch_stats)), grads = jax.value_and_grad(
        loss_fn, has_aux=True
    )(state.params)
    state = state.apply_gradients(grads=grads)
    if new_batch_stats is not None:
        state = state.replace(batch_stats=new_batch_stats)
    return state, {
        "loss": loss,
        "reconstruction_loss": loss,
        "final_loss": final_loss,
        "auxiliary_loss": auxiliary_loss,
        "psnr": psnr,
    }


@jax.jit
def eval_step(state, batch_input, batch_target):
    variables = {"params": state.params}
    if state.batch_stats is not None:
        variables["batch_stats"] = state.batch_stats
    predictions = state.apply_fn(variables, batch_input, train=False)
    reconstruction, final_loss, auxiliary_loss, final_pred = compute_supervision_losses(
        predictions,
        batch_target,
        _MODEL_CONFIGS["num_supervision_scales"],
        AUXILIARY_LOSS_WEIGHT,
    )
    final_target = resize_target_to(final_pred, batch_target)
    return {
        "loss": reconstruction,
        "reconstruction_loss": reconstruction,
        "final_loss": final_loss,
        "auxiliary_loss": auxiliary_loss,
        "sample_psnr": compute_psnr_per_example(final_pred, final_target),
        "sample_final_loss": jnp.mean(
            jnp.abs(final_pred - final_target), axis=(1, 2, 3)
        ),
    }


def train_epoch(state, train_iterator, num_scales, epoch, steps_per_epoch):
    batch_metrics = []
    task_counts = np.zeros(len(TASKS), dtype=np.int64)
    print(f"Starting training epoch {epoch}")
    for step in range(steps_per_epoch):
        batch_input, batch_target, _, task_id = next(train_iterator)
        batch_input = jnp.asarray(batch_input)
        batch_target = jnp.asarray(batch_target)
        task_ids_np = np.asarray(task_id, dtype=np.int32).reshape(-1)
        task_counts += np.bincount(task_ids_np, minlength=len(TASKS))
        rng = jax.random.fold_in(jax.random.PRNGKey(SEED + epoch), step)
        state, metrics = train_step(
            state, batch_input, batch_target, num_scales, rng
        )
        batch_metrics.append(jax.device_get(metrics))
        if (step + 1) % LOG_EVERY == 0:
            print(
                f"Epoch {epoch}, Step {step + 1}: "
                f"loss={float(metrics['loss']):.4f}, "
                f"psnr={float(metrics['psnr']):.2f} dB"
            )
    summary = {
        key: float(np.mean([np.asarray(item[key]) for item in batch_metrics]))
        for key in batch_metrics[0]
    }
    summary["task_count"] = task_counts.tolist()
    return state, summary


def evaluate(state, val_dataset, log_every=100):
    scalar_sums = {
        "loss": 0.0,
        "reconstruction_loss": 0.0,
        "final_loss": 0.0,
        "auxiliary_loss": 0.0,
    }
    task_count = np.zeros(len(TASKS), dtype=np.int64)
    task_psnr_sum = np.zeros(len(TASKS), dtype=np.float64)
    task_final_loss_sum = np.zeros(len(TASKS), dtype=np.float64)
    sample_count = 0
    start = time.time()
    print("Starting exhaustive deterministic evaluation...")
    for batch_index, (batch_input, batch_target, _, task_id) in enumerate(
        val_dataset, start=1
    ):
        metrics = jax.device_get(
            eval_step(state, jnp.asarray(batch_input), jnp.asarray(batch_target))
        )
        task_ids = np.asarray(task_id, dtype=np.int32).reshape(-1)
        sample_psnr = np.asarray(metrics["sample_psnr"], dtype=np.float64).reshape(-1)
        sample_final_loss = np.asarray(
            metrics["sample_final_loss"], dtype=np.float64
        ).reshape(-1)
        batch_size = len(task_ids)
        sample_count += batch_size
        for key in scalar_sums:
            scalar_sums[key] += float(metrics[key]) * batch_size
        task_count += np.bincount(task_ids, minlength=len(TASKS))
        np.add.at(task_psnr_sum, task_ids, sample_psnr)
        np.add.at(task_final_loss_sum, task_ids, sample_final_loss)
        if batch_index % log_every == 0:
            elapsed = time.time() - start
            print(
                f"  [eval] {batch_index} batches / {sample_count} samples | "
                f"running psnr={task_psnr_sum.sum() / sample_count:.2f} dB | "
                f"{batch_index / max(elapsed, 1e-9):.1f} batch/s"
            )
    if sample_count == 0:
        return {}
    task_psnr = np.divide(
        task_psnr_sum,
        task_count,
        out=np.full(len(TASKS), np.nan),
        where=task_count > 0,
    )
    task_final_loss = np.divide(
        task_final_loss_sum,
        task_count,
        out=np.full(len(TASKS), np.nan),
        where=task_count > 0,
    )
    summary = {key: value / sample_count for key, value in scalar_sums.items()}
    summary.update(
        {
            "psnr": float(task_psnr_sum.sum() / sample_count),
            "macro_psnr": float(np.nanmean(task_psnr)),
            "sample_count": int(sample_count),
            "task_count": task_count.tolist(),
            "task_psnr": task_psnr.tolist(),
            "task_final_loss": task_final_loss.tolist(),
        }
    )
    print(
        f"Evaluation done: {sample_count} samples | "
        f"weighted PSNR={summary['psnr']:.2f} dB | "
        f"macro PSNR={summary['macro_psnr']:.2f} dB"
    )
    for index, task in enumerate(TASKS):
        print(
            f"  {task}: n={task_count[index]}, "
            f"PSNR={task_psnr[index]:.3f} dB, "
            f"L1={task_final_loss[index]:.5f}"
        )
    return summary


# Build datasets, model and state

In [ ]:
rng = random.PRNGKey(SEED)
num_scales = _MODEL_CONFIGS["num_supervision_scales"]

# Reanudacion: none=nueva, latest=ultimo checkpoint completo,
# best=mejor PSNR validado, specific=checkpoint_<RESUME_EPOCH>.
# RESUME_EPOCH usado solamente cuando RESUME_MODE="specific"
RESUME_MODE = "latest"  # @param ["none", "latest", "best", "specific"]
RESUME_EPOCH = None      # @param {type:"integer"}

if TRAIN_MODE == "single_task":
    task_id = TASKS.index(TASK)
    train_dataset, train_size = create_dataset(
        data_dir=ACTIVE_TASK_DIRS[0],
        batch_size=BATCH_SIZE,
        patch_size=PATCH_SIZE,
        is_training=True,
    )
    val_dataset, val_size = create_dataset(
        data_dir=ACTIVE_TASK_DIRS[0],
        batch_size=EVAL_BATCH_SIZE,
        patch_size=EVAL_PATCH_SIZE,
        is_training=False,
    )
    train_dataset = add_task_id(train_dataset.unbatch(), task_id).repeat().batch(
        BATCH_SIZE, drop_remainder=True
    )
    val_dataset = add_task_id(val_dataset.unbatch(), task_id).batch(EVAL_BATCH_SIZE)
    steps_per_epoch = int(
        STEPS_PER_EPOCH_OVERRIDE
        if STEPS_PER_EPOCH_OVERRIDE is not None
        else max(1, train_size // BATCH_SIZE)
    )
    train_counts = (train_size,)
    val_counts = (val_size,)
else:
    train_dataset, steps_per_epoch, train_counts = create_unified_dataset(
        task_dirs=ACTIVE_TASK_DIRS,
        batch_size=BATCH_SIZE,
        patch_size=PATCH_SIZE,
        is_training=True,
        sampling_mode=SAMPLING_MODE_TRAIN,
    )
    val_dataset, _, val_counts = create_unified_dataset(
        task_dirs=ACTIVE_TASK_DIRS,
        batch_size=EVAL_BATCH_SIZE,
        patch_size=EVAL_PATCH_SIZE,
        is_training=False,
    )

print(f"Train counts: {train_counts}")
print(f"Validation counts: {val_counts}")
print(f"Steps per epoch: {steps_per_epoch}")
total_steps = steps_per_epoch * NUM_EPOCHS
learning_rate_fn = create_learning_rate_schedule(
    base_lr=LEARNING_RATE,
    warmup_epochs=WARMUP_EPOCHS,
    total_steps=total_steps,
    steps_per_epoch=steps_per_epoch,
)
model = build_model(ACTIVE_VARIANT)
rng, init_rng = random.split(rng)
state = create_train_state(init_rng, model, learning_rate_fn, WEIGHT_DECAY)
start_epoch = 0
best_psnr = -np.inf
best_epoch = -1
if RESUME_MODE == "none":
    existing = checkpoints.latest_checkpoint(OUTPUT_DIR)
    existing_best = checkpoints.latest_checkpoint(BEST_OUTPUT_DIR)
    if (existing or existing_best) and not ALLOW_OVERWRITE_EXISTING_RUN:
        raise FileExistsError(
            f"Existing checkpoints found in {OUTPUT_DIR}. Use RESUME_MODE='latest' "
            "or 'best', change OUTPUT_DIR, or explicitly allow overwrite."
        )
    state = initialize_state_by_mode(state)
    print("RESUME_MODE=none: starting a new run.")
else:
    resume_checkpoint = resolve_resume_checkpoint(
        OUTPUT_DIR, RESUME_MODE, RESUME_EPOCH
    )
    state, start_epoch = restore_training_state_or_fail(
        state, resume_checkpoint, steps_per_epoch
    )
    best_psnr, best_epoch = load_historical_best_psnr(OUTPUT_DIR)
    if RESUME_MODE == "best" and _checkpoint_epoch(resume_checkpoint) != best_epoch:
        raise ValueError(
            f"Best checkpoint/metadata mismatch: {resume_checkpoint}, "
            f"best_metric epoch={best_epoch}."
        )
    current_lr = float(jax.device_get(learning_rate_fn(int(state.step))))
    print(
        f"Resume OK: {resume_checkpoint} | optimizer step={int(state.step)} | "
        f"completed epochs={start_epoch} | next epoch={start_epoch + 1} | "
        f"current LR={current_lr:.3e}"
    )
    print(
        f"Historical best: {best_psnr:.6f} dB at epoch {best_epoch}."
    )
    if start_epoch >= NUM_EPOCHS:
        print(
            f"WARNING: {start_epoch} epochs are already complete; increase NUM_EPOCHS "
            "to continue training."
        )
train_dataset_iterator = iter(train_dataset)
print("Setup complete.")


Creating training dataset from /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur
Checked 8680 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur/imgs: 8680 existing, 0 missing.
Checked 8680 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/deblur/GT: 8680 existing, 0 missing.
Creating training dataset from /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze
Checked 909 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze/imgs: 909 existing, 0 missing.
Checked 909 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/dehaze/GT: 909 existing, 0 missing.
Creating training dataset from /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise
Checked 4450 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise/imgs: 4450 existing, 0 missing.
Checked 4450 files in /content/gdrive/MyDrive/Facultad/tesis/Datasets/Classifier/denoise/GT: 4450 existing, 0 missi

Resume OK: /content/gdrive/MyDrive/Facultad/tesis/ckpts/maxim_no_moe/multi_task_all_S-2_scratch/checkpoint_26 | optimizer step=52000 | completed epochs=26 | next epoch=27 | current LR=1.064e-05
Historical best: 27.271887 dB at epoch 26.
Setup complete.


# Sanity check (1-2 train steps)

In [ ]:

RUN_SANITY_CHECK = False # @param RUN_SANITY_CHECK {type: "boolean"}
if RUN_SANITY_CHECK:
    sanity_state = state
    sanity_iterator = iter(train_dataset)

    print("Running sanity check...")
    for sanity_step in range(SANITY_STEPS):
        batch_input, batch_target, _, _ = next(sanity_iterator)
        batch_input = jnp.array(batch_input)
        batch_target = jnp.array(batch_target)
        rng = jax.random.fold_in(jax.random.PRNGKey(SEED + 999), sanity_step)
        sanity_state, sanity_metrics = train_step(
            sanity_state,
            batch_input,
            batch_target,
            num_scales,
            rng,
        )
        sanity_metrics = jax.device_get(sanity_metrics)
        print(
            f"Sanity step {sanity_step + 1}/{SANITY_STEPS}: "
            f"loss={sanity_metrics['loss']:.4f}, "
            f"psnr={sanity_metrics['psnr']:.2f} dB"
        )

    print("Sanity check finished successfully.")
else:
    print("Sanity check skipped. Set RUN_SANITY_CHECK=True to run it.")

Sanity check skipped. Set RUN_SANITY_CHECK=True to run it.


# Full training loop

In [ ]:
class TeeStream:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
        return len(data)

    def flush(self):
        for stream in self.streams:
            stream.flush()


if START_TRAINING:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    console_stdout, console_stderr = sys.stdout, sys.stderr
    log_file = open(TRAIN_LOG_PATH, "a", encoding="utf-8", buffering=1)
    sys.stdout = TeeStream(console_stdout, log_file)
    sys.stderr = TeeStream(console_stderr, log_file)
    try:
        print(f"Starting comparable MAXIM baseline at {time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(
            f"S-2 | scratch={not bool(MULTITASK_INIT_CKPT)} | batch={BATCH_SIZE} | "
            f"epochs={NUM_EPOCHS} | steps/epoch={steps_per_epoch} | "
            f"train sampling={SAMPLING_MODE_TRAIN} | validation=full deterministic"
        )
        print(
            f"Training epochs {start_epoch + 1}..{NUM_EPOCHS} | "
            f"historical best={best_psnr:.6f} dB (epoch {best_epoch})"
        )
        for epoch in range(start_epoch, NUM_EPOCHS):
            real_epoch = epoch + 1
            state, train_metrics = train_epoch(
                state,
                train_dataset_iterator,
                num_scales,
                real_epoch,
                steps_per_epoch,
            )
            print(f"Epoch {real_epoch} train metrics: {train_metrics}")
            val_metrics = evaluate(state, val_dataset, log_every=100)
            print(f"Epoch {real_epoch} val metrics: {val_metrics}")
            current_psnr = float(val_metrics.get("psnr", -np.inf))
            is_best = current_psnr > best_psnr
            # Save every new best in the flat resume directory too. This keeps
            # RESUME_MODE='latest' at the last fully validated useful epoch.
            if (real_epoch % SAVE_EVERY == 0) or is_best:
                checkpoints.save_checkpoint(
                    ckpt_dir=OUTPUT_DIR,
                    target=state,
                    step=real_epoch,
                    overwrite=True,
                    keep=5,
                )
                print(f"Saved checkpoint for epoch {real_epoch} in {OUTPUT_DIR}")
            if is_best:
                best_psnr = current_psnr
                best_epoch = real_epoch
                checkpoints.save_checkpoint(
                    ckpt_dir=BEST_OUTPUT_DIR,
                    target=state,
                    step=real_epoch,
                    overwrite=True,
                    keep=1,
                )
                with open(os.path.join(BEST_OUTPUT_DIR, "best_metric.json"), "w") as handle:
                    json.dump(
                        {
                            "epoch": real_epoch,
                            "psnr": best_psnr,
                            "weighted_psnr": best_psnr,
                            "macro_psnr": float(val_metrics["macro_psnr"]),
                            "task_psnr": val_metrics["task_psnr"],
                        },
                        handle,
                    )
                print(f"Updated best checkpoint (PSNR={best_psnr:.2f})")
        print("Training finished.")
    finally:
        print(f"Complete console log saved to {TRAIN_LOG_PATH}")
        sys.stdout.flush()
        sys.stderr.flush()
        sys.stdout, sys.stderr = console_stdout, console_stderr
        log_file.close()
else:
    print(
        "Full training is disabled. The sanity check above is safe to run; "
        "set START_TRAINING=True only for the real baseline run."
    )


Starting comparable MAXIM baseline at 2026-09-15 21:14:57
S-2 | scratch=True | batch=2 | epochs=30 | steps/epoch=2000 | train sampling=uniform | validation=full deterministic
Training epochs 27..30 | historical best=27.271887 dB (epoch 26)
Starting training epoch 27
Epoch 27, Step 10: loss=0.0838, psnr=25.88 dB
Epoch 27, Step 20: loss=0.1796, psnr=37.30 dB
Epoch 27, Step 30: loss=0.1620, psnr=20.43 dB
Epoch 27, Step 40: loss=0.1216, psnr=21.19 dB
Epoch 27, Step 50: loss=0.0295, psnr=35.18 dB
Epoch 27, Step 60: loss=0.0317, psnr=40.24 dB
Epoch 27, Step 70: loss=0.0460, psnr=31.63 dB
Epoch 27, Step 80: loss=0.1228, psnr=21.85 dB
Epoch 27, Step 90: loss=0.1913, psnr=20.04 dB
Epoch 27, Step 100: loss=0.0467, psnr=34.29 dB
Epoch 27, Step 110: loss=0.0901, psnr=37.59 dB
Epoch 27, Step 120: loss=0.2304, psnr=19.81 dB
Epoch 27, Step 130: loss=0.0926, psnr=37.80 dB
Epoch 27, Step 140: loss=0.0452, psnr=41.72 dB
Epoch 27, Step 150: loss=0.0425, psnr=32.77 dB
Epoch 27, Step 160: loss=0.1030, psnr